# Task 3 - Data Augmentation

**COMP5339 Data Engineering · Assignment 1 · EV Charger Data Augmentation**

---

Augment existing DC charger records with new information from Open Charge Map (OCM) and saved PlugShare results. The notebook reads `data/interim/ev_chargers_clean.csv` from Task 2 and should be run from top to bottom. Cached source responses are reused when available.

Plug type is retrieved from both OCM and PlugShare for matched DC charger locations, standardised, and combined into the final `plug_type` field.

## Workflow

1. [OCM augmentation](#1-ocm-augmentation)
   - [Load and validate OCM data](#1-2-read-or-retrieve-the-ocm-snapshot)
   - [Clean addresses and operators](#1-3-normalise-text-and-flatten-ocm-records)
   - [Match candidates and record decisions](#1-6-apply-the-matching-rules)
   - [Deduplicate shared OCM sites](#1-10-review-shared-ocm-sites)

2. [PlugShare augmentation](#2-plugshare-augmentation)
   - [Load saved searches](#2-3-load-saved-plugshare-searches)
   - [Check DC evidence and match candidates](#2-4-inspect-plugshare-dc-evidence)
   - [Record accepted and rejected matches](#2-6-resolve-plugshare-candidates)

3. [Final dataset](#3-final-dataset)
   - [Combine plug types](#3-2-start-with-deduplicated-ocm-records)
   - [Validate and export](#3-5-save-interim-and-final-outputs)

## Outputs

- `data/raw/`: downloaded OCM and PlugShare responses.
- `data/interim/ocm_matching_review.csv` and `plugshare_matching_review.csv`: every DC record's chosen candidate, distance, evidence and decision.
- `data/interim/ev_chargers_augmented_full.csv`: the complete audit dataset — all charger fields plus every OCM and PlugShare attribute, external ID and match status. Task 4 loads this file.
- `data/processed/ev_chargers_augmented.csv`: final cleaned dataset containing the original charger fields and one combined `plug_type` field.

The final processed file excludes matching diagnostics and source-specific IDs. Those details remain in the interim audit files so every augmentation decision can be checked.


# Imports, paths and retrieval settings

Required packages: pandas, numpy, requests, python-dotenv, duckdb and IPython.
The code finds the project folder, loads Task 2 data, and selects DC records.
`OCM_REFRESH=False` reuses a verified snapshot. Set it to `True` only to retrieve an updated snapshot.
An API key is needed only for retrieval; keep `OCM_API_KEY` in `.env`.


In [1]:
%pip install -q "requests>=2.32" "python-dotenv>=1.0" "pandas>=2.2" "numpy>=1.26" "duckdb>=1.5"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import re
import math
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from difflib import SequenceMatcher
from urllib.parse import urlparse
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from dotenv import load_dotenv

# Locate the Task 2 output. The notebook runs from the project root, like the others.
project_root = Path.cwd()
clean_input = project_root / "data/interim/ev_chargers_clean.csv"
if not clean_input.exists():
    raise FileNotFoundError(
        "data/interim/ev_chargers_clean.csv is missing - run 02_data_cleaning.ipynb first."
    )

# Preserve postcodes as text and keep the full input table for the final left join.
chargers = pd.read_csv(clean_input, dtype={"postcode": "string", "sa4_code": "string"})
assert chargers["charger_id"].notna().all() and chargers["charger_id"].is_unique

# Only DC records are augmentation targets; AC rows will still survive in the final CSV.
dc_chargers = chargers.loc[chargers["charger_type"].eq("DC")].copy()
if dc_chargers.empty:
    raise ValueError("No DC records found in the Task 2 output")

# Exclude missing or implausible coordinates from the search rectangle, not from coverage.
eligible_coords = dc_chargers["latitude"].between(-37.6, -28.1) & dc_chargers["longitude"].between(
    140.9, 153.7
)
search_chargers = dc_chargers.loc[eligible_coords]
if search_chargers.empty:
    raise ValueError("No usable NSW DC coordinates")

OCM_ENDPOINT = "https://api.openchargemap.io/v3/poi/"
OCM_MAX_RESULTS = 10000
# False: reuse verified saved JSON. True: request current data and replace the cache.
# To update, change this to True and rerun sections 1.2-1.12; then set it back to False.
OCM_REFRESH = False
MATCH_RADIUS_M = 250.0
ADDRESS_SIMILARITY_MIN = 0.70
SEARCH_MARGIN_DEG = 0.02
# API filters: request Australia within the rectangle surrounding our DC sites.
# compact=false includes descriptive objects such as CurrentType and ConnectionType.
OCM_PARAMS = {
    "output": "json",
    "countrycode": "AU",
    "maxresults": OCM_MAX_RESULTS,
    "compact": "false",
    "verbose": "false",
    "boundingbox": (
        f"({search_chargers.latitude.max() + SEARCH_MARGIN_DEG},{search_chargers.longitude.min() - SEARCH_MARGIN_DEG}),"
        f"({search_chargers.latitude.min() - SEARCH_MARGIN_DEG},{search_chargers.longitude.max() + SEARCH_MARGIN_DEG})"
    ),
}


ocm_raw_dir = project_root / "data/raw/openchargemap"
plugshare_raw_dir = project_root / "data/raw/plugshare"
interim_dir = project_root / "data/interim"
processed_dir = project_root / "data/processed"
interim_dir.mkdir(parents=True, exist_ok=True)
ocm_raw_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
# Use readable filenames. Saved metadata checks whether the query filters changed.
# A refresh replaces this snapshot instead of creating another set of files.
ocm_cache = ocm_raw_dir / "locations.json"
ocm_metadata = ocm_raw_dir / "locations.metadata.json"
print(f"{len(dc_chargers)} DC records; target: {math.ceil(len(dc_chargers) / 2)} augmented records")
print("Search bounding box:", OCM_PARAMS["boundingbox"])
from IPython.display import display
import duckdb
# Add Open Charge Map api key from .env, and also Plugshare Apify api token if want to retrieve data from scratch
for env_path in [project_root / ".env", project_root.parent / ".env"]:
    if env_path.is_file():
        load_dotenv(env_path, override=False)

# Keep notebook output readable when tables contain long lists or text.
pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 45)


431 DC records; target: 216 augmented records
Search bounding box: (-28.15379,141.440104),(-36.717259000000006,153.635875)


# 1. Open Charge Map augmentation

[Open Charge Map (OCM)](https://openchargemap.io/) is an open global database of
electric-vehicle charging locations. Its API provides location coordinates,
addresses, operators, connector types, current types, power ratings, access
conditions, pricing text, and data-provenance information.

This section uses OCM as the first external augmentation source for the cleaned
TfNSW DC charger records. A local OCM snapshot is retrieved for the geographic
area containing the NSW chargers and retained in `data/raw/openchargemap` for
reproducibility.

The nested API response is flattened and standardised before matching. Candidate
locations are compared using geographic distance together with operator, street,
town, and postcode evidence. Only accepted locations with explicitly identified
DC connections contribute augmented plug types. Operator websites and clear
pricing information are also retained when available.

Every matching decision is recorded in the interim review output. Shared OCM
locations are deduplicated before the unmatched records are passed to the
PlugShare augmentation section.



## 1.1 Read or retrieve the OCM snapshot

The request uses the rectangle covering our DC coordinates, restricted to Australia.
`compact=false` keeps descriptive operator and connector objects. The saved metadata records
query parameters, retrieval time and a SHA-256 checksum, so reruns can reuse the same input.
The API limit is checked to avoid silently treating a truncated response as complete.
Transient GET failures are retried. Fresh data retrieved neeed to fill API key by signing up Open Charge Map website, aand get the key from it.

API reference: https://openchargemap.org/site/develop/api


In [3]:
def validate_ocm_payload(payload):
    """
    Validate the OCM response before it is cached or matched.

    The response must be a list of location dictionaries, and every
    location must have an OCM ID. Reject a response that reaches the API
    result limit because it may be truncated and incomplete.
    """
    # Reject unexpected JSON structures or locations without IDs.
    if not isinstance(payload, list) or not all(
        isinstance(location, dict) and location.get("ID") is not None
        for location in payload
    ):
        raise ValueError("Expected a JSON list of OCM locations with IDs")

    # Reaching the maximum suggests that more locations may be missing.
    if len(payload) >= OCM_MAX_RESULTS:
        raise ValueError(
            "OCM result limit reached; retrieve a complete snapshot before matching"
        )

    return payload


# Try the local snapshot first unless a fresh download was explicitly requested.
ocm_meta = None
if not OCM_REFRESH and ocm_cache.exists() and ocm_metadata.exists():
    try:
        meta = json.loads(ocm_metadata.read_text(encoding="utf-8"))
        body = ocm_cache.read_bytes()
        # Compare the actual query dictionary and the saved file fingerprint.
        # This verifies local data, not whether the server has newer data.
        if (
            meta["parameters"] == OCM_PARAMS
            and hashlib.sha256(body).hexdigest() == meta["sha256"]
        ):
            ocm_payload = validate_ocm_payload(json.loads(body))
            ocm_meta = meta
            print("Using saved OCM data; set OCM_REFRESH=True to download updates.")
    except (OSError, ValueError, KeyError):
        print("Cache could not be verified; a fresh retrieval is required.")

# No reusable snapshot: authenticate, request current results, and validate before saving.
# .env files were loaded explicitly in section 1.
if ocm_meta is None:
    print("Downloading a fresh OCM snapshot.")
    api_key = os.environ.get("OCM_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError("Set OCM_API_KEY in the environment or .env file ")
    with requests.Session() as ocm_session:
        retries = Retry(
            total=4,
            backoff_factor=1.5,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["GET"],
        )
        ocm_session.mount("https://", HTTPAdapter(max_retries=retries))
        ocm_session.headers.update(
            {
                "X-API-Key": api_key,
                "User-Agent": "COMP5339-Assignment1-DataAugmentation/1.0",
            }
        )
        response = ocm_session.get(OCM_ENDPOINT, params=OCM_PARAMS, timeout=(15, 120))
        response.raise_for_status()
        body = response.content
        ocm_payload = validate_ocm_payload(response.json())
    # Save to a temporary file first so interrupted writes do not replace a valid snapshot.
    temporary = ocm_cache.with_suffix(".part")
    temporary.write_bytes(body)
    temporary.replace(ocm_cache)
    # Store enough information to identify and verify this snapshot, without saving the key.
    ocm_meta = {
        "endpoint": OCM_ENDPOINT,
        "parameters": OCM_PARAMS,
        "retrieved_at": datetime.now(timezone.utc).isoformat(),
        "sha256": hashlib.sha256(body).hexdigest(),
        "records": len(ocm_payload),
    }
    temporary = ocm_metadata.with_suffix(".part")
    temporary.write_text(json.dumps(ocm_meta, indent=2), encoding="utf-8")
    temporary.replace(ocm_metadata)
    del api_key
print(f"{len(ocm_payload)} OCM records; snapshot retrieved {ocm_meta['retrieved_at']}")


Using saved OCM data; set OCM_REFRESH=True to download updates.
380 OCM records; snapshot retrieved 2026-09-20T17:53:21.915829+00:00


## 1.2 Normalise text and flatten OCM records

This section prepares the nested OCM data for matching and augmentation.

Comparison values are standardised by removing differences in capitalisation,
punctuation, repeated spaces, common street abbreviations, and state names such
as `New South Wales` and `NSW`. Original addresses and operator names are
retained for review.

Each OCM location is flattened into one row containing its identifier, address,
coordinates, operator, usage information, pricing text, website, and provenance.
Because one location can contain both AC and DC equipment, its connections are
examined individually. Only connections explicitly identified as **DC**
contribute to `ocm_plugtype`; AC connections and blank, `Unknown`, or `Other`
connector names are excluded.

In [4]:
def clean_text(value):
    """Convert a missing value to empty text; otherwise trim spaces."""
    return "" if value is None or pd.isna(value) else str(value).strip()



### Example: retrieving a DC plug type from OCM

OCM returns one JSON object per location. The notebook keeps a connector only
when its current type is explicitly DC.

```json
{
  ...
  "AddressInfo": {
    "AddressLine1": "2285 Pacific Highway",
    "Town": "Heatherbrae",
    "StateOrProvince": "NSW",
    "Postcode": "2324"
  },
  "Connections": [
    {
      "CurrentType": {
        "Title": "DC"
      },
      "ConnectionType": {
        "Title": "CCS2"
      }
    }
  ]
}

For this record:
ocm_id       = 79490
ocm_plugtype = CCS2
```
The notebook loops through Connections, keeps entries whose current type is
DC (or whose OCM current-type ID identifies DC), removes blank or placeholder
connector names, and stores the remaining connector types in ocm_plugtype.

In [5]:
# Flatten each nested API location into one row for matching and enrichment.
ocm_rows = []
# Process each OCM location
for poi in ocm_payload:
    address = poi.get("AddressInfo") or {}
    operator = poi.get("OperatorInfo") or {}
    provider = poi.get("DataProvider") or {}
    connections = poi.get("Connections") or []
    # Sites can contain both AC and DC. Keep only explicitly DC connections.
    dc_connections = []
    for connection in connections:
        current_type = connection.get("CurrentType") or {}
        if (
            clean_text(current_type.get("Title")).upper() == "DC"
            or connection.get("CurrentTypeID") == 30
        ):
            dc_connections.append(connection)
    #get list plug type
    connector_types = sorted(
        {clean_text((c.get("ConnectionType") or {}).get("Title")) for c in dc_connections}
        - {"", "Unknown", "Other"}
    )
    address_line1 = clean_text(address.get("AddressLine1"))
    address_line2 = clean_text(address.get("AddressLine2"))

    # AddressLine1 may be a site name; AddressLine2 is usually the street.
    ocm_street = address_line2 or address_line1   
    ocm_rows.append(
        {
            "ocm_id": poi["ID"],
            "ocm_title": address.get("Title"),
            "ocm_address": ", ".join(
                clean_text(address.get(k))
                for k in [
                    "AddressLine1",
                    "AddressLine2",
                    "Town",
                    "StateOrProvince",
                    "Postcode",
                ]
                if clean_text(address.get(k))
            ),
            "ocm_street": ocm_street, 
            "ocm_postcode": clean_text(address.get("Postcode")),
            "ocm_latitude": address.get("Latitude"),
            "ocm_longitude": address.get("Longitude"),
            "ocm_operator": operator.get("Title"),
            "ocm_dc_confirmed": bool(dc_connections),
            "ocm_plugtype": "; ".join(connector_types),
            "ocm_usage_type": (poi.get("UsageType") or {}).get("Title"),
            "ocm_usage_cost": poi.get("UsageCost"),
            "ocm_operator_website": operator.get("WebsiteURL"),
            "ocm_date_last_verified": poi.get("DateLastVerified"),
            "ocm_data_provider": provider.get("Title"),
            "ocm_license": provider.get("License"),
            "ocm_source_url": f"https://openchargemap.org/poi/details/{poi['ID']}",
            "ocm_retrieved_at": ocm_meta["retrieved_at"],
        }
    )
if not ocm_rows:
    raise ValueError("OCM returned no locations; inspect the request before proceeding")
ocm_locations = pd.DataFrame(ocm_rows)
if ocm_locations["ocm_id"].duplicated().any():
    raise ValueError("Repeated OCM IDs in response; review the source snapshot")
for col in ["ocm_latitude", "ocm_longitude"]:
    ocm_locations[col] = pd.to_numeric(ocm_locations[col], errors="coerce")
print(
    f"{int(ocm_locations.ocm_dc_confirmed.sum())} sites have explicitly labelled DC connections"
)


272 sites have explicitly labelled DC connections


## 1.5 Matching strategy for OCM locations

TfNSW and OCM do not share a common charger identifier, so their records cannot be
joined using an exact ID. Instead, locations are linked using coordinates together
with operator, address, town, postcode, and DC-connection evidence.

Coordinates for the same charging site are not always identical. One source may place
the point on the charger, while another may use the car-park entrance, building centre,
road access point, or manually entered GPS position. Rounding and GPS accuracy can also
move coordinates by several metres. Exact latitude and longitude equality would
therefore miss genuine matches.

Addresses can differ for similar reasons. A charger may be recorded against a shopping
centre, service station, car park, adjacent building, road entrance, or nearby street
number. Consequently, exact full-address equality is too strict.

The matching process therefore combines several pieces of evidence:
### Matching flow

For each TfNSW DC charger:

1. **Find nearby OCM candidates**
   - Calculate the distance between the TfNSW and OCM coordinates.
   - Reject candidates more than **250 metres** away.

2. **Confirm DC equipment**
   - The OCM location must contain at least one connection explicitly identified
     as DC.
   - AC-only or unknown-current locations cannot be accepted.

3. **Compare operators**
   - Standardise known aliases before comparison.
   - Examples: `BP Pulse` becomes `BP Australia`, and `Evie` becomes
     `Evie Networks`.
   - Different known operators send the candidate to review.
   - A missing operator can be allowed only for a very close candidate with a
     similar street name.

4. **Compare street evidence**
   - Ignore capitalisation, punctuation, repeated spaces, and common street
     abbreviations.
   - Street numbers are supporting evidence and are not compulsory.
   - A different or missing street number does not reject a candidate when the
     street name and other evidence agree.

5. **Check postcode evidence**
   - If both sources provide a postcode, they must match.
   - If OCM has no postcode, the candidate can still pass when:
     - the operator matches;
     - the street name is similar; and
     - the distance is no more than 250 metres.
   - A known postcode disagreement prevents automatic acceptance.

6. **Apply the distance rule**
   - **Within 50 m with the same operator:** address differences are allowed.
   - **Within 50 m with a missing operator:** require a similar street name.
   - **Between 50 m and 250 m:** require the same operator and a similar street
     name.
   - **More than 250 m:** reject.

7. **Choose one candidate**
   - If several OCM candidates pass, select the nearest passing candidate.
   - If no candidate passes, retain the nearest candidate for review with the
     reason it failed.

### Decision hierarchy

```text
Is the candidate within 250 m?
├── No  → Reject
└── Yes
    │
    ├── Does OCM confirm a DC connection?
    │   ├── No  → Review
    │   └── Yes
    │       │
    │       ├── Are both operators known but different?
    │       │   ├── Yes → Review
    │       │   └── No
    │       │       │
    │       │       ├── Are both postcodes present?
    │       │       │   ├── Yes, and different → Review
    │       │       │   └── Yes, and equal → Continue
    │       │       │
    │       │       └── Is the OCM postcode missing?
    │       │           ├── Yes → Require same operator and similar street
    │       │           └── No  → Continue
    │       │
    │       ├── Distance ≤ 50 m and operator matches
    │       │   └── Accept
    │       │
    │       ├── Distance ≤ 50 m and operator is missing
    │       │   ├── Similar street → Accept
    │       │   └── Different street → Review
    │       │
    │       └── Distance 50–250 m
    │           ├── Same operator and similar street → Accept
    │           └── Otherwise → Review

```
First we need to standardize the operator name for OCM dataset first

### 1.3.1 Standardise operator names

Use Task 2's explicit lookup approach: lowercase and normalise whitespace for lookup,
then return a canonical display name. Known aliases are listed rather than guessed by fuzzy matching.
For example, `Evie` becomes `Evie Networks`, and `BP Pulse (AU)` becomes `BP Australia`.
The table shows every OCM operator, its converted name and corresponding original names.
Generic unknown/business-owner labels provide no operator identity evidence.
Tesla variants share an operator, but the raw labels still describe different access conditions.




In [6]:
# Task 2 already cleaned the TfNSW `operator` column. Task 3 only needs to
# convert OCM and PlugShare labels to the same standard spellings.

# Build a lookup from the trusted operator names produced by Task 2.
# Example: "BP Australia" becomes {"bp australia": "BP Australia"}.
OPERATOR_LOOKUP = {
    re.sub(r"\s+", " ", str(operator)).strip().lower():
        str(operator).strip()
    for operator in chargers["operator"].dropna().unique()
}


# Known alternative names used by OCM and PlugShare.
# Each alias maps to the standard name already used by Task 2.
OPERATOR_ALIASES = {
    # BP
    "bp pulse (au)": "BP Australia",
    "bp pulse": "BP Australia",
    "bp pulse australia": "BP Australia",

    # Ampol
    "ampcharge": "Ampol",
    "ampol ampcharge": "Ampol",
    "ampol": "Ampol",

    # Evie
    "evie": "Evie Networks",
    "evie networks": "Evie Networks",

    # Tesla
    "tesla (tesla-only charging)": "Tesla",
    "tesla (including non-tesla)": "Tesla",
    "tesla supercharger": "Tesla",
    "tesla destination": "Tesla",
    "tesla destination charging": "Tesla",

    # Other observed aliases
    "evx (au)": "EVX",
    "smart charge (au)": "Smart Charge",
    "elanga (au)": "Elanga",
    "wevolt (au)": "Wevolt",
    "charge hub": "ChargeHub",
    "charge fox": "Chargefox",
    "chargefox": "Chargefox",
    "nrma": "NRMA",
    "jolt": "JOLT",
}

# Add the external aliases to the Task 2 operator lookup.
OPERATOR_LOOKUP.update(OPERATOR_ALIASES)


# These generic labels do not identify a real operator and cannot support
# operator agreement during matching.
OPERATOR_UNKNOWN = {
    "",
    "(unknown operator)",
    "(business owner at location)",
    "(private residence/individual)",
    "unknown",
    "non-networked",
    "non networked",
    "university of",
}


def match_canonical_operator(value):
    """
    Convert an OCM or PlugShare operator to the spelling used by Task 2.

    Missing and generic labels become `pd.NA`. Known aliases are converted
    through `OPERATOR_LOOKUP`. Unmapped names are retained rather than guessed.
    """
    if pd.isna(value):
        return pd.NA

    # Create a lowercase lookup key and reduce repeated whitespace.
    key = re.sub(r"\s+", " ", str(value)).strip().lower()

    if key in OPERATOR_UNKNOWN:
        return pd.NA

    return OPERATOR_LOOKUP.get(key, str(value).strip())


# ---------------------------------------------------------------------------
# Audit the conversion before changing the working OCM operator column
# ---------------------------------------------------------------------------

# Count the number of OCM locations using each original OCM operator label.
operator_audit = (
    ocm_locations
    .groupby("ocm_operator", dropna=False)
    .size()
    .reset_index(name="ocm_locations")
)

# Convert each OCM label to the Task 2 spelling.
operator_audit["operator_standardised"] = (
    operator_audit["ocm_operator"]
    .map(match_canonical_operator)
)

# Set of trusted operator names already produced by Task 2.
task2_operators = set(chargers["operator"].dropna())

# Explain how each OCM name relates to the Task 2 operator names.
operator_audit["comparison"] = operator_audit.apply(
    lambda row: (
        "missing operator evidence"
        if pd.isna(row.operator_standardised)
        else (
            "not in original dataset"
            if row.operator_standardised not in task2_operators
            else (
                "same standard spelling"
                if row.ocm_operator == row.operator_standardised
                else "alias / case variant converted"
            )
        )
    ),
    axis=1,
)

print("OCM operator-name audit:")

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_colwidth",
    100,
):
    display(operator_audit)


# Report Task 2 operators that are absent from the OCM snapshot.
ocm_standard_operators = set(
    operator_audit["operator_standardised"].dropna()
)

print("Task 2 operators not represented in OCM:")
print(sorted(task2_operators - ocm_standard_operators))


# Replace the working OCM labels only after displaying the audit.
# The original labels remain preserved in the raw OCM JSON.
ocm_locations["ocm_operator"] = (
    ocm_locations["ocm_operator"]
    .map(match_canonical_operator)
)

OCM operator-name audit:


,ocm_operator,ocm_locations,operator_standardised,comparison
0,(Business Owner at Location),8,<NA>,missing operator evidence
1,(Private Residence/Individual),1,<NA>,missing operator evidence
2,(Unknown Operator),41,<NA>,missing operator evidence
3,Ampol AmpCharge,9,Ampol,alias / case variant converted
4,BP Pulse (AU),17,BP Australia,alias / case variant converted
5,Blink Charging,1,Blink Charging,not in original dataset
6,ChargePoint,4,ChargePoint,same standard spelling
7,Chargefox,58,Chargefox,same standard spelling
8,Chargehub,1,ChargeHub,alias / case variant converted
9,EO Charging,3,EO Charging,not in original dataset


Task 2 operators not represented in OCM:
['360 EV Charge', 'AXCharge', 'Alchemy Charge', 'BMW', 'CasaCharge', 'Charge OS', 'ChargePost', 'Chargestar', 'Counties Energy', 'EV Meter', 'EVE Australia', 'EVNet', 'EVSE', 'EVUp', 'Energy Austra', 'Engie', 'Fast Cities A', 'Gentari', 'Non-networked', 'Noodoe', 'PLUS ES', 'Porsche Destination Charging', 'Porsche Smart Mobility', 'Saascharge', 'University of', 'Viva Energy Australia', 'Zeus Renewables']


### 1.5.2 Define street comparison evidence

These function compares the TfNSW `station_address` with the OCM `ocm_street`. Text is
normalized by ignoring case, punctuation, repeated spaces, and common road
abbreviations.

`SequenceMatcher` returns street similarity from `0.0` (different) to `1.0`
(identical). Candidates more than 50 m away require a score of at least **0.90**.
This is a chosen matching threshold.

Street numbers are recorded but remain optional because sources may use different
building or entrance numbers for the same site. Postcodes are checked separately.
Original addresses remain unchanged for review.


In [7]:
def normalise_key(value):
    """Ignore case and punctuation when comparing names or addresses."""
    return re.sub(r"[^a-z0-9]+", " ", clean_text(value).lower()).strip()


def normalise_address(value):
    """Normalise comparison text only; keep the original addresses for display."""
    # Lowercase both sides, replace commas/punctuation with spaces, and trim.
    key = normalise_key(value)
    # Treat the full state name and its abbreviation as the same text.
    key = re.sub(r"\bnew south wales\b", "nsw", key)
    # PlugShare commonly appends the country; OCM may omit it.
    # Remove only a trailing country token so street text is unchanged.
    key = re.sub(r"\s+ australia$", "", key)
    for long, short in {
        "street": "st",
        "road": "rd",
        "avenue": "ave",
        "highway": "hwy",
        "drive": "dr",
    }.items():
        key = re.sub(r"\b" + long + r"\b", short, key)
    return key



In [8]:
def split_street_address(value):
    """
    Split a TfNSW or external address into its leading house number and
    normalized street name for comparison.
    """
    # Use only the first comma-separated part so a later postcode is never
    # interpreted as the house number.
    text = clean_text(value).split(",")[0].strip().lower()
    # Match a leading number such as 12, 82A, 1-7 or 4/6.
    number_match = re.match(r"^(\d+[a-z]?(?:\s*[-/]\s*\d+[a-z]?)?)\s+", text)
    # Remove spaces from number ranges: "1 - 7" becomes "1-7".
    number = re.sub(r"\s+", "", number_match.group(1)) if number_match else ""
    # Remove the leading house number before comparing street names.
    name = text[number_match.end() :] if number_match else text
    name = normalise_address(name)
    # Remove trailing suburb/state text when a normal street suffix is present.
    # A complex address without a recognised suffix remains conservative for review.
    street = re.match(
        r"^(.+?\b(?:st|rd|ave|hwy|dr|lane|ln|pde|parade|crescent|cres|way|place|pl|court|ct|terrace|tce))\b",
        name,
    )
    return number, street.group(1) if street else name


In [9]:
def address_evidence(charger, candidate, distance):
    """
    Compare one TfNSW charger with one OCM candidate and return the evidence.

    This function calculates street-number agreement, street-name similarity,
    postcode conflict, distance, and DC confirmation. It records evidence only;
    `match_compare()` makes the final accept, review, or reject decision.
    """
    # Split both addresses into an optional house number and normalized street.
    source_number, source_name = split_street_address(charger["station_address"])
    candidate_number, candidate_name = split_street_address(candidate["ocm_street"])
     # Street numbers can be compared only when both sources provide one.
    both_numbered = bool(source_number and candidate_number)
    street_number_match = both_numbered and source_number == candidate_number
    street_number_conflict = both_numbered and source_number != candidate_number
    # Compare only street names because house numbers are optional evidence.
    street_name_similarity = (
        SequenceMatcher(None, source_name, candidate_name).ratio()
        if source_name and candidate_name
        else 0.0
    )

    source_postcode = clean_text(charger["postcode"])
    candidate_postcode = clean_text(candidate["ocm_postcode"])
    postcode_conflict = bool(
        source_postcode and candidate_postcode and source_postcode != candidate_postcode
    )

    return {
        "candidate_ocm_id": candidate["ocm_id"],
        "ocm_title": candidate["ocm_title"],
        "ocm_street": candidate["ocm_street"],
        "ocm_operator": candidate["ocm_operator"],
        "ocm_postcode": candidate["ocm_postcode"],
        "ocm_latitude": candidate["ocm_latitude"],
        "ocm_longitude": candidate["ocm_longitude"],
        "distance_m": float(distance),
        "street_number_match": street_number_match,
        "source_street_number": source_number,
        "ocm_street_number": candidate_number,
        "street_number_conflict": street_number_conflict,
        "street_name_similarity": street_name_similarity,
        "postcode_conflict": postcode_conflict,
        "dc_confirmed": bool(candidate["ocm_dc_confirmed"]),
    }


### 1.5.3  OCM matching rules function

For each TfNSW-OCM candidate pair, this step combines distance, operator,
street, postcode, and DC evidence into a final decision.

Candidates more than 250 m away are rejected. Close candidates require operator
or street agreement, while candidates between 50 and 250 m require both the
same operator and a similar street name. Matching postcodes are required when
both are present; a missing OCM postcode is allowed only with strong operator,
street, and distance evidence. OCM must also confirm a DC connection.

The result records the evidence, final status (`accept`, `review`, or `reject`),
and a readable reason for the decision.

In [23]:
def match_operator(value):
    """
    Convert an operator name to a simple comparison key.

    Known aliases are first converted to the standard Task 2 name. Case and
    punctuation are then removed so equivalent names compare consistently.
    """
    return normalise_key(
        match_canonical_operator(value)
    )


def match_compare(charger, candidate, distance):
    """
    Decide whether candidate matches one TfNSW DC charger.

    The initial decision uses distance, operator agreement, and street-name
    similarity. Postcode and DC evidence are then applied as final checks.
    The returned dictionary contains the evidence, decision, and reason.
    """

    # Collect street, postcode, distance, and DC evidence calculated by the
    # separate evidence function.
    evidence = address_evidence(
        charger,
        candidate,
        distance,
    )

    # Standardize both operator names before comparing them.
    source_operator = match_operator(
        charger["operator"]
    )
    candidate_operator = match_operator(
        candidate["ocm_operator"]
    )

    # These generic values do not provide reliable operator evidence.
    unknown_operators = {
        "",
        "unknown",
        "non networked",
        "university of",
        "other",
    }

    # Operator evidence is missing if either side has no meaningful value.
    operator_missing = (
        source_operator in unknown_operators
        or candidate_operator in unknown_operators
    )

    # Operators agree only when both values are meaningful and equal.
    operator_same = (
        not operator_missing
        and source_operator == candidate_operator
    )

    source_number = evidence["source_street_number"]
    candidate_number = evidence["ocm_street_number"]

    # Remove leading zeros before comparing numbers.
    # Example: "01" and "1" both become "1".
    def number_key(value):
        return re.sub(
            r"\d+",
            lambda match: str(int(match.group())),
            value,
        )

    both_numbered = bool(
        source_number and candidate_number
    )
    street_numbers_match = (
        both_numbered and number_key(source_number) == number_key(candidate_number)
    )

    # Street names are considered similar when the normalized similarity score
    # calculated by `address_evidence()` is at least 0.90.
    street_similar = (
        evidence["street_name_similarity"] >= 0.90
    )

    # -----------------------------------------------------------------------
    # distance, operator, and street evidence
    # -----------------------------------------------------------------------

    # Candidates farther than 250 metres are outside the matching tolerance.
    if distance > 250:
        decision = "reject"
        reason = "distance exceeds 250 m"

    # Two different known operators prevent automatic acceptance.
    elif not operator_missing and not operator_same:
        decision = "reject"
        reason = "different known operators"

    # Within 50 metres, coordinate and operator agreement are strong enough
    # even when the recorded addresses differ.
    elif distance <= 50 and operator_same:
        decision = "accept"
        reason = (
            "within 50 m and same/aliased operator; "
            "address disagreement allowed"
        )

    # A missing operator can be tolerated within 50 metres only when the
    # normalized street names are similar.
    elif distance <= 50 and operator_missing and street_similar:
        decision = "accept"
        reason = (
            "within 50 m; operator missing but address similar"
        )

    # Between 50 and 250 metres, require both operator and street agreement.
    elif operator_same and street_similar:
        decision = "accept"
        reason = (
            "50–250 m with same/aliased operator and similar street name; "
            "house number optional"
        )

    # All other candidates lack enough evidence for automatic acceptance.
    else:
        decision = "review"
        reason = (
            "insufficient operator/address agreement for this distance"
        )


    # -----------------------------------------------------------------------
    # \postcode evidence
    # -----------------------------------------------------------------------

    source_postcode = clean_text(
        charger["postcode"]
    )
    candidate_postcode = clean_text(
        candidate["ocm_postcode"]
    )

    # A postcode match requires both values to be present and equal.
    postcode_match = bool(
        source_postcode
        and candidate_postcode
        and source_postcode == candidate_postcode
    )

    # A missing OCM postcode is allowed only when the source postcode exists,
    # operators agree, street names are similar, and distance is within 250 m.
    postcode_missing_exception = bool(
        source_postcode
        and not candidate_postcode
        and operator_same
        and street_similar
        and distance <= 250
    )

    # Record postcode evidence for review.
    evidence["postcode_match"] = postcode_match
    evidence["postcode_missing_exception"] = (
        postcode_missing_exception
    )

    if decision == "accept" and postcode_missing_exception:
        reason += (
            "; OCM postcode missing; operator, address and distance "
            "support match"
        )

    # Downgrade an initially accepted candidate when postcode evidence is
    # insufficient or the two known postcodes disagree.
    if decision == "accept" and not (
        postcode_match or postcode_missing_exception
    ):
        decision = "review"

        reason += "; " + (
            "postcode missing on one or both sides"
            if not source_postcode or not candidate_postcode
            else "postcodes disagree"
        )

    # -----------------------------------------------------------------------
    # Explicit DC evidence
    # -----------------------------------------------------------------------

    # OCM must explicitly identify at least one DC connection before its
    # attributes can be attached to a TfNSW DC charger.
    if decision == "accept" and not evidence["dc_confirmed"]:
        decision = "review"
        reason += "; DC connection not confirmed"

    # -----------------------------------------------------------------------
    # Store the final evidence and decision
    # -----------------------------------------------------------------------

    evidence.update(
        operator_match=operator_same,
        operator_relation=(
            "missing"
            if operator_missing
            else "same/alias"
            if operator_same
            else "different"
        ),
        street_number_match=street_numbers_match,
        street_number_conflict=(
            both_numbered and not street_numbers_match
        ),
        address_similar=street_similar,
        rule_decision=decision,
        street_number_policy="optional_supporting_evidence",
        candidate_passes=(decision == "accept"),
        candidate_reason=reason,
        ocm_address=candidate["ocm_address"],
    )

    return evidence

### 1.5.4 Calculate candidate distances with DuckDB Spatial

DuckDB Spatial calculates the WGS84 spheroidal distance, in **metres**, between
each TfNSW DC charger and OCM location. The function uses
`POINT_2D(latitude, longitude)` coordinates.

All OCM candidates within 250 m are retained for matching. The nearest candidate
outside this range is also retained when necessary so unmatched records can still
be inspected. Distance only generates and orders candidates; it does not prove a
match.



In [24]:
#check valid geographic values
valid_ocm = ocm_locations.loc[
    ocm_locations.ocm_latitude.between(-90, 90)
    & ocm_locations.ocm_longitude.between(-180, 180)
].reset_index(drop=True)

# DuckDB Spatial builds the candidate list and measures distances in metres.
# The DataFrames are registered as SQL views; no extra database file is needed.
import duckdb

with duckdb.connect() as spatial_con:
    try:
        spatial_con.execute("LOAD spatial")
    except duckdb.Error:
        # Only install if not already available (installation needs network access).
        spatial_con.execute("INSTALL spatial")
        spatial_con.execute("LOAD spatial")

    spatial_con.register("dc_input", dc_chargers)
    spatial_con.register("ocm_input", valid_ocm)

    spatial_candidates = spatial_con.execute(
        """
        WITH dc_points AS (
            SELECT charger_id,
                   ST_Point(latitude, longitude)::POINT_2D AS point
            FROM dc_input
            WHERE latitude BETWEEN -37.6 AND -28.1
              AND longitude BETWEEN 140.9 AND 153.7
        ), ocm_points AS (
            SELECT ocm_id,
                   ST_Point(ocm_latitude, ocm_longitude)::POINT_2D AS point
            FROM ocm_input
        ), distances AS (
            SELECT d.charger_id, o.ocm_id,
                   ST_Distance_Spheroid(d.point, o.point) AS distance_m
            FROM dc_points AS d
            CROSS JOIN ocm_points AS o
        ), ranked AS (
            SELECT *, ROW_NUMBER() OVER (
                PARTITION BY charger_id ORDER BY distance_m
            ) AS distance_rank
            FROM distances
        )
        SELECT charger_id, ocm_id, distance_m, distance_rank
        FROM ranked
        WHERE distance_m <= ? OR distance_rank = 1
        ORDER BY charger_id, distance_m, ocm_id
    """,
        [MATCH_RADIUS_M],
    ).df()


### 1.5.5 Choose a candidate for each charger

Apply the rules to every candidate. If several pass, select the nearest; exact distance ties
use the external ID for a deterministic choice. Record the passing-candidate count and reason. If none pass, retain the nearest candidate and its failure reason.
Every original DC record receives a review row, including records with unusable coordinates.


In [25]:
def choose_candidate(candidates, id_field):
    """Return the nearest passing candidate, or nearest candidate for review."""
    candidates = list(candidates)
    passing = [row for row in candidates if row["candidate_passes"]]
    pool = passing or candidates
    chosen = min(pool, key=lambda row: (row["distance_m"], str(row[id_field])))
    return chosen, len(passing)


# Look up the SQL results by charger ID, then apply the existing text checks.
spatial_by_charger = {
    charger_id: group.to_dict("records")
    for charger_id, group in spatial_candidates.groupby("charger_id")
}
ocm_by_id = valid_ocm.set_index("ocm_id", drop=False)

match_rows = [] #compact matching result.
review_rows = [] #selected candidate and detailed reason.
candidate_rows = [] #evidence for every candidate evaluated.
for charger in dc_chargers.to_dict("records"):
    # One review row per source charger; source and candidate fields stay distinct.
    review = {
        "charger_id": charger["charger_id"],
        "station_name": charger["station_name"],
        "station_address": charger["station_address"],
        "operator": charger["operator"],
        "postcode": charger["postcode"],
        "latitude": charger["latitude"],
        "longitude": charger["longitude"],
        "match_status": "unmatched",
        "match_reason": "no OCM locations with usable coordinates",
        "candidate_role": "none",
    }
    match = {
        "charger_id": charger["charger_id"],
        "match_status": "unmatched",
        "ocm_id": None,
        "match_distance_m": None,
    }
    lat, lon = charger["latitude"], charger["longitude"]
    if not (
        pd.notna(lat)
        and pd.notna(lon)
        and -37.6 <= lat <= -28.1
        and 140.9 <= lon <= 153.7
    ):
        review.update(
            match_status="invalid_coordinates",
            match_reason="source coordinates missing or outside NSW range",
        )
        match["match_status"] = review["match_status"]
    elif charger["charger_id"] in spatial_by_charger:
        evidence = []
        for spatial_candidate in spatial_by_charger[charger["charger_id"]]:
            candidate = ocm_by_id.loc[spatial_candidate["ocm_id"]]
            item = match_compare(charger, candidate, spatial_candidate["distance_m"])
            evidence.append(item)
            candidate_rows.append({"charger_id": charger["charger_id"], **item})
        chosen, passing_count = choose_candidate(
            evidence, "candidate_ocm_id"
        )

        review.update(chosen)
        review["passing_candidates"] = passing_count

        if passing_count:
            review.update(
                match_status="accepted",
                match_reason=chosen["candidate_reason"]
                + (
                    f"; nearest of {passing_count} passing candidates"
                    if passing_count > 1
                    else ""
                ),
                candidate_role="accepted_match",
            )
            match.update(
                match_status="accepted",
                ocm_id=chosen["candidate_ocm_id"],
                match_distance_m=chosen["distance_m"],
            )
        else:
            review.update(
                match_status=chosen["rule_decision"],
                match_reason=chosen["candidate_reason"],
                candidate_role="review_only_not_accepted",
            )
            match["match_status"] = review["match_status"]
    match_rows.append(match)
    review_rows.append(review)

match_matches = pd.DataFrame(match_rows)
match_review = pd.DataFrame(review_rows)
match_candidates = pd.DataFrame(candidate_rows)


#### OCM match preview
Show five accepted and five non-accepted records for a quick check; the full review remains in the CSV.


In [26]:
preview_columns = [
    "charger_id", "station_address", "operator",
    "ocm_street", "ocm_operator", "distance_m",
    "postcode_match", "match_status", "match_reason",
]
accepted_preview = match_review.loc[
    match_review.match_status.eq("accepted"), preview_columns
].head(5)
unmatched_preview = match_review.loc[
    ~match_review.match_status.eq("accepted"), preview_columns
].head(5)
print("Accepted OCM matches (first 5):")
display(accepted_preview.round({"distance_m": 1}))
print("Unmatched/review OCM records (first 5):")
display(unmatched_preview.round({"distance_m": 1}))


Accepted OCM matches (first 5):


,charger_id,station_address,operator,ocm_street,ocm_operator,distance_m,postcode_match,match_status,match_reason
0,2,"01 Wallgrove Road, Sydney, 2766",BP Australia,Wallgrove Road & Old Wallgrove Road,BP Australia,65.2,True,accepted,50–250 m with same/aliased operator and s...
7,24,"1 Ingham Dr, Casula NSW 2170",Evie Networks,1 Ingham Dr,Evie Networks,125.4,True,accepted,50–250 m with same/aliased operator and s...
9,36,"1 Park St, Sydney, 2103",JOLT,1 Park St,JOLT,5.7,True,accepted,within 50 m and same/aliased operator; ad...
16,60,"10 Winery Dr, Port Macquarie, 2444",Tesla,764 Pacific Highway,Tesla,0.1,True,accepted,within 50 m and same/aliased operator; ad...
19,69,"1067 Oxley Hwy, Thrumster, 2444",Chargefox,1063 Oxley Highway,Chargefox,29.9,True,accepted,within 50 m and same/aliased operator; ad...


Unmatched/review OCM records (first 5):


,charger_id,station_address,operator,ocm_street,ocm_operator,distance_m,postcode_match,match_status,match_reason
1,3,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",NRMA,82 Marsh Street,NRMA,29.2,False,review,within 50 m and same/aliased operator; ad...
2,6,"1 Bay St, Sydney, 2037",Tesla,1 Bay Street,Tesla,19.5,False,review,within 50 m and same/aliased operator; ad...
3,7,"1 Bells Blvd, Kingscliff, 2487",Evie Networks,45 Cabarita Road,NRMA,6196.5,False,reject,distance exceeds 250 m
4,12,"1 Dalgal Way, Sydney, 2037",Tesla,1 Bay Street,Tesla,1673.9,False,reject,distance exceeds 250 m
5,17,"1 Frederick St, Sydney, 2064",Evie Networks,65 Reserve Road,Chargefox,156.7,False,reject,different known operators


### 1.5.6 Inspect matched and unmatched records

Read both addresses and operators alongside distance, postcode evidence and the decision reason.
These are automated rule decisions, not independent verification of physical station identity.
The same information is saved in `data/interim/ocm_matching_review.csv`.


In [27]:
#preview of records that still need PlugShare matching.
unmatched = match_review.loc[~match_review.match_status.eq("accepted")]
print(f"Unmatched/review records: {len(unmatched)}")
display(unmatched.head(5))


Unmatched/review records: 309


,charger_id,station_name,station_address,operator,postcode,latitude,longitude,match_status,match_reason,candidate_role,candidate_ocm_id,ocm_title,ocm_street,ocm_operator,ocm_postcode,...,street_number_conflict,street_name_similarity,postcode_conflict,dc_confirmed,postcode_match,postcode_missing_exception,operator_match,operator_relation,address_similar,rule_decision,street_number_policy,candidate_passes,candidate_reason,ocm_address,passing_candidates
1,3,NaN,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",NRMA,2836,-30.511874,151.669395,review,within 50 m and same/aliased operator; ad...,review_only_not_accepted,170824,Armidale Visitors Centre,82 Marsh Street,NRMA,2350,...,True,0.666667,True,True,False,False,True,same/alias,False,review,optional_supporting_evidence,False,within 50 m and same/aliased operator; ad...,"82 Marsh Street, Armidale, New South Wale...",0
2,6,NaN,"1 Bay St, Sydney, 2037",Tesla,2037,-33.883504,151.194433,review,within 50 m and same/aliased operator; ad...,review_only_not_accepted,311308,Tesla Destination Charger,1 Bay Street,Tesla,2007,...,False,1.000000,True,False,False,False,True,same/alias,True,review,optional_supporting_evidence,False,within 50 m and same/aliased operator; ad...,"1 Bay Street, Broadway, NSW, 2007",0
3,7,NaN,"1 Bells Blvd, Kingscliff, 2487",Evie Networks,2487,-28.276903,153.577078,reject,distance exceeds 250 m,review_only_not_accepted,295994,NRMA Cabarita Beach,45 Cabarita Road,NRMA,2488,...,True,0.285714,True,True,False,False,False,different,False,reject,optional_supporting_evidence,False,distance exceeds 250 m,"45 Cabarita Road, Bogangar, New South Wal...",0
4,12,NaN,"1 Dalgal Way, Sydney, 2037",Tesla,2037,-33.876501,151.178000,reject,distance exceeds 250 m,review_only_not_accepted,100404,Broadway Supercharger,1 Bay Street,Tesla,2007,...,False,0.250000,True,True,False,False,True,same/alias,False,reject,optional_supporting_evidence,False,distance exceeds 250 m,"1 Bay Street, Broadway, 2007",0
5,17,NaN,"1 Frederick St, Sydney, 2064",Evie Networks,2064,-33.817079,151.188937,reject,different known operators,review_only_not_accepted,272613,TLE Electrical Artarmon,65 Reserve Road,Chargefox,2065,...,True,0.454545,True,True,False,False,False,different,False,reject,optional_supporting_evidence,False,different known operators,"65 Reserve Road, St Leonards, New South W...",0


### 1.5.7 Review and deduplicate shared OCM sites

Task 2 reconciled exact and conflicting duplicates using the fields available in
the original TfNSW dataset. However, some probable duplicates could not be identified
during cleaning because their coordinates or address text were not exactly equal.

Inspection against OCM revealed that several TfNSW records independently selected the
same accepted OCM location. Many of these records have the same or equivalent address
but slightly different latitude and longitude values. This can occur when coordinates
refer to different chargers, parking bays, entrances, or manually recorded GPS points
within the same physical station. Exact coordinate comparison therefore does not always
identify duplicate source records.

The opposite issue also occurs: two source records can have identical coordinates but
different address descriptions. Coordinates alone are therefore not sufficient evidence
for either retaining or removing a record.

After selecting one OCM candidate for each TfNSW charger, this section performs a
cross-record check. It identifies accepted source records that selected the same
`candidate_ocm_id`. This check does not alter the earlier distance, operator, address,
postcode, or DC decisions; it examines relationships between records only after those
individual matches have passed.

When several accepted source records select the same OCM location, they are treated as
duplicate representations of that external station. One source record is retained using
the following order:

1. Keep the record with the most non-null original TfNSW fields.
2. If completeness is equal, keep the record closest to the OCM coordinate.

The remaining records are removed from the deduplicated OCM review, PlugShare target
set, coverage denominator, and final processed dataset. The complete shared-site table
is displayed before removal so each `KEEP` and `REMOVE` decision remains reviewable.

This second deduplication stage complements Task 2 rather than replacing it. Task 2
uses only internal TfNSW evidence, while Task 3 uses agreement with an independent
external location to reveal duplicates whose source coordinates or address formatting
differ.

In [28]:
accepted_ocm = match_review.loc[
    match_review.match_status.isin(["accepted", "shared_site_review"])
    & match_review.candidate_ocm_id.notna()
].copy()
shared_ids = accepted_ocm.candidate_ocm_id.value_counts()
shared_ids = set(shared_ids[shared_ids > 1].index)
shared_ocm = accepted_ocm.loc[accepted_ocm.candidate_ocm_id.isin(shared_ids)].copy()
if shared_ocm.empty:
    dedup_keep_ids, dedup_remove_ids = set(chargers.charger_id), set()
    print("No shared accepted OCM sites found.")
else:
    shared_ocm["has_street_number"] = shared_ocm.station_address.map(
        lambda value: bool(re.match(r"^\s*\d+", clean_text(value)))
    ).astype(int)
    shared_ocm["source_non_null"] = (
        shared_ocm[[c for c in chargers.columns if c in shared_ocm.columns]]
        .notna()
        .sum(axis=1)
    )
    # For a confirmed shared-site duplicate, retain the most complete original
    # source record. If completeness ties, retain the record closest to OCM.
    keep = shared_ocm.sort_values(
        ["candidate_ocm_id", "source_non_null", "distance_m"],
        ascending=[True, False, True],
    ).drop_duplicates("candidate_ocm_id")
    dedup_keep_ids = set(keep.charger_id.astype(int)) | set(
        chargers.charger_id
    ) - set(shared_ocm.charger_id.astype(int))
    dedup_remove_ids = set(shared_ocm.charger_id.astype(int)) - set(
        keep.charger_id.astype(int)
    )
    print("Keep:", sorted(dedup_keep_ids & set(shared_ocm.charger_id.astype(int))))
    print("Remove:", sorted(dedup_remove_ids))

    # Show every shared duplicate and mark the row retained by the rule.
    shared_display = shared_ocm[[
        "candidate_ocm_id", "charger_id", "station_address", "operator",
        "ocm_address", "distance_m", "has_street_number", "source_non_null"
    ]].copy()
    shared_display["decision"] = shared_display.charger_id.astype(int).map(
        lambda value: "KEEP" if value in dedup_keep_ids else "REMOVE"
    )
    display(shared_display.sort_values(["candidate_ocm_id", "decision", "charger_id"]))



# Remove confirmed shared-site duplicates before saving the OCM review. The
# saved file is also the input used by the standalone PlugShare retriever.
match_review = match_review.loc[
    ~match_review.charger_id.isin(dedup_remove_ids)
].copy()
match_review.to_csv(interim_dir / "ocm_matching_review.csv", index=False)
print(
    " OCM review:",
    (interim_dir / "ocm_matching_review.csv").relative_to(project_root),
)




Keep: [141, 409, 821, 829, 872, 1074]
Remove: [142, 187, 307, 410, 873, 1691]


,candidate_ocm_id,charger_id,station_address,operator,ocm_address,distance_m,has_street_number,source_non_null,decision
145,126130,409,"29-55 Twynam St, Narrandera, 2700",NRMA,"31 Twynam Street, Narrandera, NSW, 2700",16.951842,1,6,KEEP
146,126130,410,"29-55 Twynam St, Narrandera, 2700",NRMA,"31 Twynam Street, Narrandera, NSW, 2700",204.117967,1,6,REMOVE
46,190670,141,"134 Lachlan St, Hay, 2711",NRMA,"134 Lachlan St, Hay, NSW, 2711",27.349568,1,6,KEEP
47,190670,142,"134 Lachlan St, Hay, 2711",NRMA,"134 Lachlan St, Hay, NSW, 2711",174.436644,1,6,REMOVE
364,266501,1074,20-22 Camden Rd Campbelltown NSW 2560 Aus...,Tesla,"20-22 Camden Road, Campbelltown, New Sout...",88.191566,1,6,KEEP
105,266501,307,20-22 Camden Rd Campbelltown NSW 2560 Aus...,Tesla,"20-22 Camden Road, Campbelltown, New Sout...",130.761830,1,6,REMOVE
290,272619,821,"Bunnerong Rd, Sydney, 2036",Chargefox,"439 Bunnerong Road, Chifley, New South Wa...",86.723864,0,6,KEEP
397,272619,1691,801-899R Bunnerong Rd Chifley NSW 2036 Au...,Chargefox,"439 Bunnerong Road, Chifley, New South Wa...",86.731791,1,6,REMOVE
314,273428,872,"Hassall St, Sydney, 2142",BP Australia,"Hassall Street, Rosehill, New South Wales...",76.120257,0,6,KEEP
315,273428,873,"Hassall Street & James Rouse Drive, Sydne...",BP Australia,"Hassall Street, Rosehill, New South Wales...",85.382865,0,6,REMOVE


 OCM review: data\interim\ocm_matching_review.csv


### 1.5.8 Build the retained OCM augmentation table

Remove the shared-site records rejected above, join accepted OCM attributes by external ID,
and retain every Task 2 row for the later final integration.


In [29]:
# Keep the compact OCM match table consistent with the deduplicated review.
match_matches = match_matches.loc[
    ~match_matches.charger_id.isin(dedup_remove_ids)
].copy()

# Attach the complete OCM row through the accepted external ID.
augmentation = match_matches.merge(
    ocm_locations,
    on="ocm_id",
    how="left",
    validate="many_to_one",
)
augmentation["augmented"] = augmentation["ocm_id"].notna()

# Keep every original Task 2 row at this stage. Confirmed duplicate IDs are
# removed in the final integration after PlugShare deduplication is complete.
augmented = chargers.merge(
    augmentation,
    on="charger_id",
    how="left",
    validate="one_to_one",
)
augmented["match_status"] = augmented["match_status"].fillna("not_targeted")
augmented["augmented"] = (
    augmented["augmented"].astype("boolean").fillna(False).astype(bool)
)

assert len(augmented) == len(chargers)
assert augmented.charger_id.is_unique


# 2. PlugShare augmentation

[PlugShare](https://www.plugshare.com/) is a community-maintained directory of EV charging
locations. Its station records can include coordinates, addresses, charging networks, DC outlets,
connector types, cost indicators, and operator service URLs.

This project retrieves PlugShare data through an Apify scraper because PlugShare does not provide
the required bulk records through the OCM API. Responses are cached in `data/raw/plugshare` to avoid
repeating paid and time-consuming requests. Only DC records left unmatched after OCM are searched.

PlugShare applies the same matching methodology as OCM; it does not use a separate or looser
matching rule. The workflow validates the local source data, flattens nested station items,
confirms DC evidence, calculates distances with DuckDB Spatial, applies the shared operator/address/
postcode rules, choose the nearest passing candidate, and review shared external locations.


## 2.1 Read or retrieve the PlugShare cache

Retrieval is handled by `tools/Plug_share.py`. If
`data/raw/plugshare/coordinate_searches.json` exists, the notebook reuses it without making an API
call. If it is absent, the script uses the local Apify token to search around each OCM-unmatched DC
charger. Each search uses the original coordinates, a 1 km search radius, and up to two results.


In [30]:
plugshare_cache_path = plugshare_raw_dir / 'coordinate_searches.json'
if plugshare_cache_path.exists():
    ps_cache = json.loads(plugshare_cache_path.read_text(encoding='utf-8'))
    print(f'PlugShare JSON found: {plugshare_cache_path.relative_to(project_root)}')
    print(f"Saved searches: {len(ps_cache.get('searches', {}))}")
    display(pd.DataFrame([
        {'address': entry.get('address'), 'charger_ids': entry.get('charger_ids'),
         'status': entry.get('status'), 'results': len(entry.get('items', []))}
        for entry in ps_cache.get('searches', {}).values()
    ]))
else:
    # Retrieve automatically through the reusable Python module.
    import sys
    sys.path.insert(0, str(project_root))
    from tools import Plug_share
    Plug_share.main()
    ps_cache = json.loads(plugshare_cache_path.read_text(encoding='utf-8'))


PlugShare JSON found: data\raw\plugshare\coordinate_searches.json
Saved searches: 309


,address,charger_ids,status,results
0,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",[3],SUCCEEDED,2
1,"1 Bay St, Sydney, 2037",[6],SUCCEEDED,2
2,"1 Bells Blvd, Kingscliff, 2487",[7],SUCCEEDED,2
3,"1 Dalgal Way, Sydney, 2037",[12],SUCCEEDED,2
4,"1 Frederick St, Sydney, 2064",[17],SUCCEEDED,2
...,...,...,...,...
304,"William St, Sydney, 2046",[982],SUCCEEDED,2
305,"charger 973: University of Wollongong, No...",[973],SUCCEEDED,2
306,"charger 974: University of Wollongong, No...",[974],SUCCEEDED,2
307,188 Fernleigh Rd Glenfield Park NSW 2650 ...,[1320],SUCCEEDED,2


## 2.2 Prepare the PlugShare target records

Build the PlugShare target set from DC records not accepted by OCM and not removed as OCM
duplicates. Load only the saved coordinate-search JSON. Every returned item is inspected rather
than assuming the first API result is correct.


In [35]:
ps_targets = dc_chargers.loc[
    ~dc_chargers.charger_id.isin(
        set(match_matches.loc[match_matches.match_status.eq("accepted"), "charger_id"])
        | dedup_remove_ids
    )
].copy()
ps_target_ids = set(ps_targets.charger_id)
print(
    f'{len(ps_targets)} DC records remain after OCM; {len(ps_cache["searches"])} saved PlugShare searches.'
)


309 DC records remain after OCM; 309 saved PlugShare searches.


## 2.3 Flatten records and confirm DC evidence

Flatten each nested PlugShare item into one candidate row. A candidate has DC evidence when an
outlet explicitly reports `isDc=true` or `powerType="DC"`. If outlet details are absent,
`isFastCharger=true` is accepted as fallback DC evidence, but no plug type is inferred.

Only explicitly DC outlets contribute connector types, network operators, and operator URLs.
Postcodes are extracted from the Australian address, and known network aliases use the same
standard operator names as OCM and TfNSW. The raw JSON remains unchanged.

A simplified saved PlugShare item looks like this:

```json
{
  "id": 479757,
  "name": "bp Eastern Creek",
  "address": "1 Wallgrove Rd, Eastern Creek NSW 2766, Australia",
  "latitude": -33.810983,
  "longitude": 150.850275,
  "isFastCharger": true,
  "cost": true,
  "outlets": [
    {
      "isDc": true,
      "powerType": "DC",
      "connectorType": "CCS2",
      "network": {
        "name": "bp pulse",
        "action_url": "https://www.bp.com/en_au/australia/home/products-services/bppulse/"
      }
    }
  ],
  "plugshareUrl": "https://www.plugshare.com/location/479757"
}
```

The flattening step converts this nested item into one candidate row containing the PlugShare ID,
name, address, postcode, coordinates, canonical operator, confirmed-DC flag, `CCS2` plug type,
cost indicator, operator website and source URL.


In [36]:
from urllib.parse import urlparse
def website_domain(value):
    """Return a lowercase hostname without `www.` or any URL path."""
    text = clean_text(value)
    if not text:
        return pd.NA
    if re.match(r"^https?://", text, flags=re.I):
        hostname = (urlparse(text).hostname or "").lower()
    elif re.fullmatch(r"(?:www\.)?[a-z0-9.-]+\.[a-z]{2,}", text, flags=re.I):
        hostname = text.lower()
    else:
        return pd.NA
    hostname = re.sub(r"^www\.", "", hostname)
    return hostname if hostname else pd.NA


def ps_operator(value):
    return match_canonical_operator(value)


def ps_postcode(address):
    match = re.search(
        r"\b(?:NSW|New South Wales|ACT|VIC|QLD|SA|WA|TAS|NT)\s+(\d{4})\b",
        clean_text(address),
        flags=re.I,
    )
    return match.group(1) if match else ""


ps_pairs = {}
for entry in ps_cache["searches"].values():
    if not entry.get("download_complete"):
        continue
    for item in entry.get("items", []):
        if item.get("id") is None:
            continue
        outlets = item.get("outlets") or []
        dc_outlets = [
            o
            for o in outlets
            if o.get("isDc") is True or clean_text(o.get("powerType")).upper() == "DC"
        ]
        dc_confirmed = bool(dc_outlets) or (
            not outlets and item.get("isFastCharger") is True
        )
        plugs = sorted(
            {clean_text(o.get("connectorType")) for o in dc_outlets}
            - {"", "Unknown", "Other"}
        )
        networks = [(o.get("network") or {}).get("name") for o in dc_outlets]
        # PlugShare stores an operator service link in each outlet network.
        # Keep it only when the DC outlets agree on one URL.
        operator_urls = sorted(
            {
                website_domain(
                    (outlet.get("network") or {}).get("action_url")
                    or (outlet.get("network") or {}).get("actionUrl")
                    or (outlet.get("network") or {}).get("url")
                )
                for outlet in dc_outlets
            }
            - {"", pd.NA}
        )
        if not any(networks):
            fallback = item.get("networkNames") or []
            networks = [fallback] if isinstance(fallback, str) else fallback
        operators = sorted({clean_text(ps_operator(v)) for v in networks} - {""})
        for charger_id in set(entry.get("charger_ids", [])) & ps_target_ids:
            # Deduplicate the same location returned by repeated searches for one source record.
            ps_pairs[(charger_id, str(item["id"]))] = {
                "charger_id": charger_id,
                "plugshare_id": str(item["id"]),
                "plugshare_name": item.get("name"),
                "plugshare_address": item.get("address"),
                "plugshare_latitude": item.get("latitude"),
                "plugshare_longitude": item.get("longitude"),
                "plugshare_operators": operators,
                "plugshare_postcode": ps_postcode(item.get("address")),
                "dc_confirmed": dc_confirmed,
                "dc_plug_types": "; ".join(plugs),
                "plugshare_operator_website": (
                    operator_urls[0] if len(operator_urls) == 1 else pd.NA
                ),
                "plugshare_cost": item.get("cost"),
                "plugshare_cost_description": item.get("costDescription"),
                "plugshare_url": item.get("plugshareUrl"),
                "retrieved_at": entry.get("downloaded_at"),
            }
ps_candidates = pd.DataFrame(
    ps_pairs.values(),
    columns=[
        "charger_id",
        "plugshare_id",
        "plugshare_name",
        "plugshare_address",
        "plugshare_latitude",
        "plugshare_longitude",
        "plugshare_operators",
        "plugshare_postcode",
        "dc_confirmed",
        "dc_plug_types",
        "plugshare_operator_website",
        "plugshare_cost",
        "plugshare_cost_description",
        "plugshare_url",
        "retrieved_at",
    ],
)
for col in ["plugshare_latitude", "plugshare_longitude"]:
    ps_candidates[col] = pd.to_numeric(ps_candidates[col], errors="coerce")
print(
    f"{len(ps_candidates)} source/candidate pairs; {int(ps_candidates.dc_confirmed.sum())} have DC evidence."
)


579 source/candidate pairs; 388 have DC evidence.


## 2.4 Calculate distances and apply the shared rules

DuckDB Spatial measures each PlugShare candidate from the original TfNSW coordinate in metres.
PlugShare fields are adapted to the structure expected by `match_compare()`. Both sources therefore
use the same distance, operator, street, postcode, and DC requirements. A missing
PlugShare postcode follows the same restricted exception as a missing OCM postcode.


In [37]:
with duckdb.connect() as ps_con:
    ps_con.execute("LOAD spatial")
    ps_con.register("ps_input", ps_candidates)
    ps_con.register("source_input", ps_targets)
    ps_distances = ps_con.execute(
        """
        SELECT p.charger_id, p.plugshare_id,
               ST_Distance_Spheroid(ST_Point(s.latitude,s.longitude)::POINT_2D,
                   ST_Point(p.plugshare_latitude,p.plugshare_longitude)::POINT_2D) AS distance_m
        FROM ps_input p JOIN source_input s USING (charger_id)
        WHERE s.latitude BETWEEN -90 AND 90 AND s.longitude BETWEEN -180 AND 180
          AND p.plugshare_latitude BETWEEN -90 AND 90 AND p.plugshare_longitude BETWEEN -180 AND 180
    """
    ).df()
ps_candidates = ps_candidates.merge(
    ps_distances, on=["charger_id", "plugshare_id"], how="left", validate="one_to_one"
)
ps_source = ps_targets.set_index("charger_id")
ps_evidence = []
for row in ps_candidates.to_dict("records"):
    source = ps_source.loc[row["charger_id"]]
    operators = row["plugshare_operators"]
    source_operator = clean_text(match_canonical_operator(source["operator"]))
    operator = source_operator if source_operator in operators else "; ".join(operators)
    adapter = {
        "ocm_id": row["plugshare_id"],
        "ocm_title": row["plugshare_name"],
        "ocm_street": row["plugshare_address"],
        "ocm_address": row["plugshare_address"],
        "ocm_operator": operator,
        "ocm_postcode": row["plugshare_postcode"],
        "ocm_latitude": row["plugshare_latitude"],
        "ocm_longitude": row["plugshare_longitude"],
        "ocm_dc_confirmed": row["dc_confirmed"],
    }
    if pd.isna(row["distance_m"]):
        evidence = {
            "candidate_passes": False,
            "rule_decision": "review",
            "candidate_reason": "missing/invalid coordinates",
        }
    else:
        # Compare street portions consistently on both sides; full addresses remain in the review.
        adapter["ocm_street"] = clean_text(row["plugshare_address"]).split(",")[0]
        evidence = match_compare(source, adapter, row["distance_m"])
    reason = evidence["candidate_reason"].replace("OCM", "PlugShare")
    ps_evidence.append(
        {
            **row,
            "station_address": source["station_address"],
            "operator": source["operator"],
            "postcode": source["postcode"],
            "plugshare_operator": "; ".join(operators),
            **{
                key: evidence.get(key)
                for key in [
                    "operator_relation",
                    "address_similar",
                    "postcode_match",
                    "postcode_conflict",
                    "postcode_missing_exception",
                    "candidate_passes",
                    "rule_decision",
                ]
            },
            "candidate_reason": reason,
        }
    )
ps_evidence = pd.DataFrame(ps_evidence)


## 2.5 Choose one candidate for each charger

The shared `choose_candidate()` function selects the nearest candidate that passes every rule. If
none pass, the nearest failed candidate is retained for review. External ID breaks exact distance
ties. Missing or failed searches remain unmatched, and every target charger receives one review row.


In [38]:
ps_rows = []
for source in ps_targets.to_dict("records"):
    candidates = (
        ps_evidence.loc[ps_evidence.charger_id.eq(source["charger_id"])]
        if not ps_evidence.empty
        else pd.DataFrame()
    )
    row = {
        "charger_id": source["charger_id"],
        "station_address": source["station_address"],
        "operator": source["operator"],
        "postcode": source["postcode"],
        "match_status": "unmatched",
        "match_reason": "no downloaded candidates",
        "search_has_dc_item": False,
    }
    if not candidates.empty:
        chosen, passing_count = choose_candidate(
            candidates.to_dict("records"), "plugshare_id"
        )

        row.update(chosen)
        row["search_has_dc_item"] = bool(candidates.dc_confirmed.any())
        row["match_status"] = (
            "accepted" if passing_count else chosen["rule_decision"]
        )
        row["match_reason"] = chosen["candidate_reason"] + (
            f"; nearest of {passing_count} passing candidates"
            if passing_count > 1
            else ""
        )
        row["passing_candidates"] = passing_count
    ps_rows.append(row)
ps_review = pd.DataFrame(ps_rows)
if "plugshare_id" not in ps_review:
    ps_review["plugshare_id"] = pd.NA
# Match each source independently; a PlugShare site may support several source records.
# Flag shared PlugShare IDs; retain all source records.
accepted_ids = ps_review.loc[ps_review.match_status.eq("accepted"), "plugshare_id"]
shared_plugshare_ids = set(accepted_ids.dropna().value_counts().loc[lambda values: values > 1].index)
ps_review["shared_plugshare_site_flag"] = ps_review["plugshare_id"].isin(shared_plugshare_ids)
ps_review.to_csv(interim_dir / "plugshare_matching_review.csv", index=False)
ps_columns = [
    "charger_id",
    "station_address",
    "operator",
    "plugshare_address",
    "plugshare_operator",
    "distance_m",
    "postcode",
    "plugshare_postcode",
    "search_has_dc_item",
    "dc_confirmed",
    "dc_plug_types",
    "match_status",
    "match_reason",
]
print(ps_review.match_status.value_counts().to_string())
with pd.option_context("display.max_rows", None, "display.max_colwidth", 100):
    print("Accepted PlugShare matches:")
    display(
        ps_review.loc[ps_review.match_status.eq("accepted")]
        .reindex(columns=ps_columns)
        .head(5)
    )
    print("Unmatched / review:")
    display(
        ps_review.loc[~ps_review.match_status.eq("accepted")]
        .reindex(columns=ps_columns)
        .head(5)
    )


match_status
accepted     213
review        47
reject        46
unmatched      3
Accepted PlugShare matches:


,charger_id,station_address,operator,plugshare_address,plugshare_operator,distance_m,postcode,plugshare_postcode,search_has_dc_item,dc_confirmed,dc_plug_types,match_status,match_reason
2,7,"1 Bells Blvd, Kingscliff, 2487",Evie Networks,"Shop/3 Bells Blvd, Kingscliff NSW 2487, Australia",Evie Networks,6.644511,2487,2487,True,True,CCS2; CHAdeMO,accepted,within 50 m and same/aliased operator; address disagreement allowed
4,17,"1 Frederick St, Sydney, 2064",Evie Networks,"1 Frederick St, Artarmon NSW 2064, Australia",Evie Networks,48.048290,2064,2064,True,True,CCS2,accepted,within 50 m and same/aliased operator; address disagreement allowed
5,20,"1 Greenbridge Dr, Wilton, 2571",Chargefox,"1 Greenbridge Dr, Wilton NSW 2571, Australia",Chargefox,3.253084,2571,2571,True,True,CCS2,accepted,within 50 m and same/aliased operator; address disagreement allowed
7,37,"1 Perry St, Batemans Bay, 2536",Evie Networks,"15 Clyde St, Batemans Bay NSW 2536, Australia",Evie Networks,45.553781,2536,2536,True,True,CCS2,accepted,within 50 m and same/aliased operator; address disagreement allowed
8,38,"1 Plunkett St, Sydney, 2065",Exploren,"Plunkett St near Cnr 32 Chandos St, St Leonards NSW 2065, Australia",Exploren,30.188780,2065,2065,True,True,CCS2,accepted,within 50 m and same/aliased operator; address disagreement allowed


Unmatched / review:


,charger_id,station_address,operator,plugshare_address,plugshare_operator,distance_m,postcode,plugshare_postcode,search_has_dc_item,dc_confirmed,dc_plug_types,match_status,match_reason
0,3,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",NRMA,"81 Marsh St, Armidale NSW 2350, Australia",NRMA,27.160117,2836,2350,True,True,CCS2; CHAdeMO,review,within 50 m and same/aliased operator; address disagreement allowed; postcodes disagree
1,6,"1 Bay St, Sydney, 2037",Tesla,"Basement/1 Bay St, Ultimo NSW 2007, Australia",,33.684068,2037,2007,True,False,,review,insufficient operator/address agreement for this distance
3,12,"1 Dalgal Way, Sydney, 2037",Tesla,"1 Dalgal Wy, Forest Lodge NSW 2037, Australia",Supercharger,20.095755,2037,2037,True,True,CCS2,reject,different known operators
6,28,"1 Little Walker St, Casino, NSW 2470, Australia",NRMA,"15 Victory St, Braidwood NSW 2622, Australia",NRMA,5.370469,2470,2622,True,True,CCS2; CHAdeMO,review,within 50 m and same/aliased operator; address disagreement allowed; postcodes disagree
9,48,"1 Wharf St, Tweed Heads, 2485",Tesla,"Tweed Mall, corner, Wharf St, Tweed Heads NSW 2485, Australia",Supercharger,11.284035,2485,2485,True,True,CCS2,reject,different known operators


## 2.6 Review and deduplicate shared PlugShare sites

After individual matching, identify accepted source records that selected the same PlugShare ID.
These represent one external station. Keep the source row with the most non-null original fields;
if completeness ties, keep the row closest to the PlugShare coordinate. Display every `KEEP` and
`REMOVE` decision before the removed IDs are excluded from final integration.


In [39]:
# Start from accepted PlugShare matches that have a real external location ID.
accepted_ps = ps_review.loc[
    ps_review.match_status.eq("accepted")
    & ps_review.plugshare_id.notna()
].copy()

# Every accepted group sharing one PlugShare ID represents one external site.
# Keep exactly one source record from each shared group.
shared_id_counts = accepted_ps.plugshare_id.value_counts()
shared_plugshare_ids = set(shared_id_counts[shared_id_counts > 1].index)
shared_ps = accepted_ps.loc[
    accepted_ps.plugshare_id.isin(shared_plugshare_ids)
].copy()

plugshare_dedup_keep_ids = set()
plugshare_dedup_remove_ids = set()

if shared_ps.empty:
    print("No shared accepted PlugShare sites found.")
else:
    # Prefer the source row containing the most original information. If rows
    # are equally complete, retain the one closest to the PlugShare coordinate;
    # charger_id provides a stable final tie-breaker.
    source_columns = [
        column for column in chargers.columns if column in shared_ps.columns
    ]
    shared_ps["source_non_null"] = shared_ps[source_columns].notna().sum(axis=1)

    keep = shared_ps.sort_values(
        ["plugshare_id", "source_non_null", "distance_m", "charger_id"],
        ascending=[True, False, True, True],
    ).drop_duplicates("plugshare_id")

    plugshare_dedup_keep_ids = set(keep.charger_id.astype(int))
    plugshare_dedup_remove_ids = (
        set(shared_ps.charger_id.astype(int)) - plugshare_dedup_keep_ids
    )

    shared_ps["decision"] = shared_ps.charger_id.astype(int).map(
        lambda charger_id: (
            "KEEP" if charger_id in plugshare_dedup_keep_ids else "REMOVE"
        )
    )

    print(f"Shared accepted PlugShare sites: {len(shared_plugshare_ids)}")
    print("Keep:", sorted(plugshare_dedup_keep_ids))
    print("Remove:", sorted(plugshare_dedup_remove_ids))

    shared_display = shared_ps[[
        "plugshare_id",
        "charger_id",
        "station_address",
        "operator",
        "plugshare_address",
        "distance_m",
        "source_non_null",
        "decision",
    ]].copy()
    display(
        shared_display.sort_values(
            ["plugshare_id", "decision", "charger_id"]
        )
    )

ps_review["plugshare_duplicate_removed_flag"] = (
    ps_review.charger_id.isin(plugshare_dedup_remove_ids)
)


Shared accepted PlugShare sites: 14
Keep: [68, 175, 248, 265, 399, 513, 521, 577, 665, 758, 773, 862, 936, 1399]
Remove: [50, 176, 178, 779, 863, 1341, 1439, 1519, 1555, 1618, 1637, 1723, 1771, 1848]


,plugshare_id,charger_id,station_address,operator,plugshare_address,distance_m,source_non_null,decision
102,1009229,399,"2-8 Turramurra Ave, Sydney, 2074",Evie Networks,"2 - 8 Turramurra Ave, Turramurra NSW 2074",11.761241,4,KEEP
265,1009229,1439,2-8 Turramurra Ave Turramurra NSW 2074 Au...,Evie Networks,"2 - 8 Turramurra Ave, Turramurra NSW 2074",18.082105,4,REMOVE
195,1110425,773,"9 Vickery Ave, Sydney, 2029",Evie Networks,"9 Vickery Ave, Rose Bay NSW 2029, Australia",6.322851,4,KEEP
285,1110425,1723,9 Vickery Ave Rose Bay NSW 2029 Australia,Evie Networks,"9 Vickery Ave, Rose Bay NSW 2029, Australia",106.396413,4,REMOVE
221,519090,862,"Gardeners Rd, Sydney, 2020",JOLT,"904 Gardeners Rd, Mascot NSW 2020, Australia",7.951562,4,KEEP
...,...,...,...,...,...,...,...,...
42,886692,178,"15 Ridge St, Sydney, 2060",Evie Networks,"47 Ridge St, North Sydney NSW 2060, Austr...",60.665427,4,REMOVE
67,920863,265,"2 Coast Hospital Rd, Sydney, 2036",Evie Networks,"1 Coast Hospital Rd, Little Bay NSW 2036,...",7.961673,4,KEEP
261,920863,1341,2 Coast Hospital Rd Little Bay NSW 2036 A...,Evie Networks,"1 Coast Hospital Rd, Little Bay NSW 2036,...",29.846971,4,REMOVE
190,922708,758,"88-90 Houston Rd, Sydney, 2032",Evie Networks,"88-90P Houston Rd, Kingsford NSW 2032, Au...",0.092467,4,KEEP


# 3. Final dataset

This section combines accepted attributes, validates the row set, and saves interim audit data and the compact processed CSV.


## 3.1 Combine the final augmented attributes

Create three analysis-ready attributes for matched DC chargers:

- `plug_type`: standardized connector names from OCM or PlugShare;
- `operator_website`: a valid OCM operator URL when available;
- `cost_applies`: nullable Boolean (`True` = cost indicated, `False` = explicitly free,
  missing = the source does not establish either).

OCM pricing is free text, so only explicit free or paid wording is classified. PlugShare supplies a
structured `cost` flag. Source-specific values and pricing text remain in the interim audit output.
Coverage for each final attribute is measured against the deduplicated DC records.


In [40]:
def standard_plugs(value):
    """Return unique connector labels using the same canonical spelling."""
    aliases = {
        "CCS (Type 2)": "CCS2",
        "CCS (Type 1)": "CCS1",
        "CHAdeMO": "CHAdeMO",
    }
    return "; ".join(
        sorted(
            {
                aliases.get(part.strip(), part.strip())
                for part in clean_text(value).split(";")
                if part.strip()
            }
        )
    )


WEBSITE_DOMAIN_ALIASES = {
    "joltcharge.com": "jolt.com.au",
    "teslamotors.com": "tesla.com",
}


def clean_website(value):
    """Return one canonical HTTPS operator website without paths or legacy aliases."""
    domain = website_domain(value)
    if pd.isna(domain):
        return pd.NA
    canonical_domain = WEBSITE_DOMAIN_ALIASES.get(domain, domain)
    return f"https://{canonical_domain}"


def classify_ocm_cost(value):
    """Convert clear OCM pricing text to paid/free; leave uncertain text missing."""
    text = clean_text(value).lower()
    if not text:
        return pd.NA
    zero_cost = re.fullmatch(
        r"\$?\s*0+(?:\.0+)?(?:\s*(?:aud|dollars?|cents?|/\s*kwh|per\s+kwh))?",
        text,
    )
    if re.search(r"\bfree\b|no (?:charge|cost|fee)|complimentary", text) or zero_cost:
        return False
    if re.search(
        r"\$\s*\d|\d+(?:\.\d+)?\s*(?:c|¢)\s*/?\s*kwh|"
        r"per\s+kwh|/kwh|charging fee|payment required|fees? apply|paid charging",
        text,
    ):
        return True
    return pd.NA


def nullable_boolean(value):
    """Preserve True, False and missing instead of treating missing as False."""
    if pd.isna(value):
        return pd.NA
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    text = str(value).strip().lower()
    if text == "true":
        return True
    if text == "false":
        return False
    return pd.NA


### 3.2 Initialise attributes from accepted OCM matches

Start with the records remaining after OCM and PlugShare duplicate decisions. Standardize OCM plug
types, validate operator URLs, and classify only explicit OCM cost statements. Missing or unclear
pricing remains missing rather than being incorrectly labelled free.


In [41]:
# Remove records rejected by either source's duplicate resolution.
all_remove_ids = set(dedup_remove_ids) | set(plugshare_dedup_remove_ids)
combined = augmented.loc[~augmented.charger_id.isin(all_remove_ids)].copy()

# Initialise the consolidated attributes from accepted OCM matches.
combined["ocm_plugtype"] = combined["ocm_plugtype"].map(standard_plugs)
combined["plug_type"] = combined["ocm_plugtype"].replace(r"^\s*$", pd.NA, regex=True)
combined["plug_type_source"] = np.where(
    combined.plug_type.notna(), "Open Charge Map", pd.NA
)

combined["operator_website"] = combined["ocm_operator_website"].map(clean_website)
combined["operator_website_source"] = np.where(
    combined.operator_website.notna(), "Open Charge Map", pd.NA
)

combined["cost_applies"] = combined["ocm_usage_cost"].map(
    classify_ocm_cost
).astype("boolean")
combined["cost_source"] = np.where(
    combined.cost_applies.notna(), "Open Charge Map", pd.NA
)


### 3.3 Add accepted PlugShare attributes

Join only accepted, retained PlugShare matches. PlugShare fills plug type and `cost_applies` for
records not matched to OCM. Its original `costDescription` remains in the interim audit data.


In [42]:
ps_accepted = ps_review.loc[
    ps_review.match_status.eq("accepted")
    & ~ps_review.charger_id.isin(plugshare_dedup_remove_ids)
].copy()
assert ps_accepted.charger_id.is_unique
assert not set(ps_accepted.charger_id) & set(
    match_matches.loc[match_matches.match_status.eq("accepted"), "charger_id"]
)

ps_add = ps_accepted.reindex(
    columns=[
        "charger_id",
        "plugshare_id",
        "dc_plug_types",
        "plugshare_operator_website",
        "plugshare_cost",
        "plugshare_cost_description",
        "plugshare_url",
        "retrieved_at",
    ]
).rename(
    columns={
        "dc_plug_types": "plugshare_plugtype",
        "retrieved_at": "plugshare_retrieved_at",
    }
)
ps_add["plugshare_cost"] = ps_add["plugshare_cost"].map(
    nullable_boolean
).astype("boolean")

combined = combined.merge(
    ps_add,
    on="charger_id",
    how="left",
    validate="one_to_one",
)

# PlugShare targets only OCM-unmatched records, so these values fill gaps rather
# than overwrite accepted OCM attributes.
ps_has_plugs = combined.plugshare_plugtype.fillna("").ne("")
combined.loc[ps_has_plugs, "plug_type"] = combined.loc[
    ps_has_plugs, "plugshare_plugtype"
].map(standard_plugs)
combined.loc[ps_has_plugs, "plug_type_source"] = "PlugShare"

# Fill the operator URL from PlugShare when its accepted DC outlets provide one
# unambiguous network action URL. OCM values retain priority if present.
ps_has_website = (
    combined.operator_website.isna()
    & combined.plugshare_operator_website.map(clean_website).notna()
)
combined.loc[ps_has_website, "operator_website"] = combined.loc[
    ps_has_website, "plugshare_operator_website"
].map(clean_website)
combined.loc[ps_has_website, "operator_website_source"] = "PlugShare"

ps_has_cost = combined.plugshare_cost.notna()
combined.loc[ps_has_cost, "cost_applies"] = combined.loc[
    ps_has_cost, "plugshare_cost"
]
combined.loc[ps_has_cost, "cost_source"] = "PlugShare"
combined["cost_applies"] = combined["cost_applies"].astype("boolean")


### 3.4 Validate the combined attributes

Normalize missing values, record the final match status, and count a row as augmented when at least
one requested final attribute is known. Preserve all original Task 2 values for retained records.


In [43]:
# Normalize blanks before calculating final coverage.
combined["plug_type"] = combined["plug_type"].replace(r"^\s*$", pd.NA, regex=True)
combined["operator_website"] = combined["operator_website"].replace(
    r"^\s*$", pd.NA, regex=True
)
combined["plugshare_match_status"] = combined.charger_id.map(
    ps_review.set_index("charger_id").match_status
).fillna("not_targeted")
combined["overall_match_status"] = np.where(
    combined.plugshare_id.notna(), "accepted", combined.match_status
)

# A DC record is augmented when any requested final attribute is available.
combined["augmented"] = (
    combined.charger_type.eq("DC")
    & (
        combined.plug_type.notna()
        | combined.operator_website.notna()
        | combined.cost_applies.notna()
    )
)

assert not set(combined.charger_id) & all_remove_ids
assert len(combined) == len(chargers) - len(all_remove_ids)
assert combined.charger_id.is_unique
expected_source = chargers.loc[
    ~chargers.charger_id.isin(all_remove_ids)
].copy()
pd.testing.assert_frame_equal(
    combined[chargers.columns].sort_values("charger_id").reset_index(drop=True),
    expected_source.sort_values("charger_id").reset_index(drop=True),
    check_dtype=False,
)


### 3.5 Save interim and final outputs
Write the complete audit file to `data/interim` and the compact dataset with one `plug_type` column to `data/processed`.


In [44]:
IMPORTANT_FINAL_COLUMNS = [
    'charger_id', 'site_id', 'station_name', 'station_address', 'lga_name', 'postcode',
    'latitude', 'longitude', 'sa4_code', 'sa4_name', 'gcc_code', 'gcc_name', 'state_name',
    'operator', 'charger_type', 'charger_status', 'number_of_plugs', 'power_kw_min',
    'power_kw_max', 'connectors_in_rating', 'rating_format', 'plug_type',
'operator_website', 'cost_applies'
]

final_columns = [column for column in IMPORTANT_FINAL_COLUMNS if column in combined.columns]
# Select the public output columns and give the combined plug field its final name.
final_clean = combined[final_columns].copy()


In [45]:
# Save the compact final dataset and the complete audit version.
final_clean.to_csv(processed_dir / "ev_chargers_augmented.csv", index=False)
combined.to_csv(interim_dir / "ev_chargers_augmented_full.csv", index=False)

# Measure each requested attribute against the deduplicated DC population.
final_dc_mask = combined.charger_type.eq("DC")
final_dc_count = int(final_dc_mask.sum())
coverage_rows = []
for attribute in ["plug_type", "operator_website", "cost_applies"]:
    available = int((final_dc_mask & combined[attribute].notna()).sum())
    coverage_rows.append(
        {
            "attribute": attribute,
            "DC records": available,
            "DC total": final_dc_count,
            "coverage_pct": round(100 * available / final_dc_count, 2),
        }
    )

any_attribute_count = int((final_dc_mask & combined.augmented).sum())
coverage_rows.append(
    {
        "attribute": "any requested attribute",
        "DC records": any_attribute_count,
        "DC total": final_dc_count,
        "coverage_pct": round(100 * any_attribute_count / final_dc_count, 2),
    }
)

coverage_summary = pd.DataFrame(coverage_rows)
display(coverage_summary)
print("Saved combined dataset: data/processed/ev_chargers_augmented.csv")


,attribute,DC records,DC total,coverage_pct
0,plug_type,315,411,76.64
1,operator_website,312,411,75.91
2,cost_applies,284,411,69.10
3,any requested attribute,315,411,76.64


Saved combined dataset: data/processed/ev_chargers_augmented.csv


In [46]:
final_clean

,charger_id,site_id,station_name,station_address,lga_name,postcode,latitude,longitude,sa4_code,sa4_name,gcc_code,gcc_name,state_name,operator,charger_type,charger_status,number_of_plugs,power_kw_min,power_kw_max,connectors_in_rating,rating_format,plug_type,operator_website,cost_applies
0,1,1,NaN,"Muswellbrook, 2333",Muswellbrook Shire Council,2333,-32.262242,150.890139,106,Hunter Valley exc Newcastle,1RNSW,Rest of NSW,New South Wales,EVUp,AC,Operational,2,22.0,22.0,NaN,number+unit,<NA>,<NA>,<NA>
1,2,2,NaN,"01 Wallgrove Road, Sydney, 2766",Blacktown City Council,2766,-33.811004,150.849597,116,Sydney - Blacktown,1GSYD,Greater Sydney,New South Wales,BP Australia,DC,Operational,4,150.0,150.0,NaN,number+unit,CCS2,https://bp.com,True
2,3,3,NaN,"1 - 7 Ross St, Wilcannia NSW 2836, Australia",Central Darling Shire Council,2836,-30.511874,151.669395,110,New England and North West,1RNSW,Rest of NSW,New South Wales,NRMA,DC,Operational,4,50.0,50.0,NaN,number+unit,<NA>,<NA>,<NA>
3,4,4,NaN,"1 Balfour St, Sydney, 2070",Ku-ring-gai Council,2070,-33.774101,151.167035,121,Sydney - North Sydney and Hornsby,1GSYD,Greater Sydney,New South Wales,Chargefox,AC,Operational,7,22.0,22.0,NaN,number+unit,<NA>,<NA>,<NA>
4,5,5,NaN,"1 Bay Ln, Byron Bay, 2481",Byron Shire Council,2481,-28.641819,153.613633,112,Richmond - Tweed,1RNSW,Rest of NSW,New South Wales,Tesla,AC,Operational,2,19.0,19.0,NaN,number+unit,<NA>,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1921,1942,1931,NaN,17 Dulwich St Dulwich Hill NSW 2203 Austr...,Inner West Council,2203,-33.902552,151.140843,120,Sydney - Inner West,1GSYD,Greater Sydney,New South Wales,PLUS ES,AC,Operational,1,7.0,7.0,NaN,number+unit,<NA>,<NA>,<NA>
1922,1943,1932,NaN,17 Flood St Bondi NSW 2026 Australia,Waverley Council,2026,-33.891058,151.258787,118,Sydney - Eastern Suburbs,1GSYD,Greater Sydney,New South Wales,EVX,AC,Operational,2,22.0,22.0,NaN,number+unit,<NA>,<NA>,<NA>
1923,1944,1933,NaN,17 Grove St Dulwich Hill NSW 2203 Australia,Inner West Council,2203,-33.901739,151.139465,120,Sydney - Inner West,1GSYD,Greater Sydney,New South Wales,PLUS ES,AC,Operational,1,7.0,7.0,NaN,number+unit,<NA>,<NA>,<NA>
1924,1945,1934,NaN,17 Hereward St Maroubra NSW 2035 Australia,Randwick City Council,2035,-33.945104,151.256166,118,Sydney - Eastern Suburbs,1GSYD,Greater Sydney,New South Wales,PLUS ES,AC,Operational,1,22.0,22.0,NaN,number+unit,<NA>,<NA>,<NA>
